# Session 1 — Optimal Growth — Course Notebook

**Goals**
1. Load the `OptimalGrowth` package (local `.py` module).
2. Simulate a baseline economy.
3. Compute and *display* the planner's **effective optimization window**.
4. Provide **exercises** with empty cells for students to complete.
5. Run **sensitivity analyses** (parameters **and** the planner's window size).

> This notebook is self‑contained and uses the local file `OptimalGrowth.py` provided with the course materials.


## Session 1 — Objectives and prerequisites

**Learning objectives**
- Run the original `OptimalGrowth.py` model and identify its states, controls, and welfare objective.
- Recover the baseline optimal saving path.
- Perform controlled comparative statics for depreciation and capital share.

**Prerequisites.** Slides for Session 1; constrained optimization intuition; NumPy arrays and Matplotlib.


## Technical setup

Run from this session directory. Install dependencies in the next cell; imports follow separately.

**Dates.** Row zero contains only initial states (capital, productivity, population). Output, consumption, investment and saving are `NaN` there. Row j stores flows and the saving decision over the interval ending at j, using states from row j-1. These missing initial flows are intentional and must not be replaced by zero.

Open this notebook together with the current OptimalGrowth.py from session1. After downloading updated files, restart the kernel and run all cells from the top. The import cell prints the module path so you can check which copy is used.


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


## 1) Setup & Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
import OptimalGrowth as og

# Reload the module if this notebook was already open when the file changed.
importlib.reload(og)
print("Model file used:", og.__file__)


## 2) Model Parameters (`Params`)

Baseline shared with the slides: alpha = 0.30, delta = 0.06, rho = 0.015, gamma = 2, A = L = 1, K0 = 3, Delta = 1 year. The exercises below change one parameter at a time.


In [ ]:
# Inspect default parameters
p = og.Params()
print(p)
print("\nNumber of periods nT =", p.nT, "with step Δ =", p.Delta, "years")
print("Time runs from", p.t0, "to", p.tT)

## 3) Initialize state and determine exogenous path

Before solving the planner’s problem, we need to **set the environment of the economy**.  
This involves two steps:

1. **Exogenous variables** — external drivers such as productivity and population, which follow paths we specify in advance.  
2. **Initial state variables** — endogenous stocks like capital, which determine where the economy starts from.

We begin with the exogenous variables.

### Setting exogenous states

Exogenous state variables are given by $x_t \in \mathbb{R}^{N_x}$: productivity $A_t$, population $L_t$.

They follow an exogenous process:
$$
x_t = g(x_{t-1}), \quad g: \mathbb{R}^{N_x} \to \mathbb{R}^{N_x}.
$$

In this simplified notebook, we use two exogenous variables, $$A_t=L_t=1$$ that we have **fixed both to 1** (flat paths) to isolate the core mechanics.  
We store these in the $y$ matrix later; other variables are left as `NaN` until solved.

### Setting initial state variables
We also initialize state variables.


In [ ]:
# Initialize the simulation matrix with exogenous drivers and initial states
sim = og.init_states(p)

# Convert to a tidy DataFrame for inspection (optional)
df0 = og.mat_to_df(sim, p)
df0.head()

## 4) The planner's optimization problem

Given initial capital, productivity and population paths, choose saving to maximize discounted utility. This is a deterministic problem (there is no expectation operator).

$$
\max_{s_0,\ldots,s_{H-1}}\sum_{t=0}^{H-1}\beta^t L_t u(\kappa C_t/L_t),\qquad \beta=(1+\rho)^{-\Delta},\quad \kappa=1000.
$$

The constraints are $Y_t=A_t K_t^\alpha L_t^{1-\alpha}$, $C_t=(1-s_t)Y_t$, $I_t=s_tY_t$, and $K_{t+1}=(1-\delta)^\Delta K_t+\Delta I_t$, with $K_0$ given and $0\le s_t<1$. Utility is $u(c)=(c^{1-\gamma}-1)/(1-\gamma)$ (or $\log c$ when $\gamma=1$).

The implementation uses bound-constrained optimization, a small consumption floor, and a rolling finite window with zero terminal payoff. Its discount threshold determines the window, but is not by itself a bound on welfare truncation error. See **growth_model.pdf** for the compact problem, Euler equation and the exact mapping between mathematical dates and array rows.


In [ ]:
# Initialize the model and choose the decision dates.
sim = og.init_states(p)
timevec = np.arange(1, p.nT)
bounds = (0.0, 1.0)
control_id = [p.i_s]

# Compute the optimal saving path.
sim_opt = og.run_optimal_policy(sim, timevec, p, bounds, control_id)

# Reproduce the planner-window calculation, step by step.
discount = 1.0
T_planner = 1
while discount > p.toly:
    T_planner = T_planner + 1
    discount = discount * (1.0 / (1.0 + p.rho)) ** p.Delta

print("Planning window (periods):", T_planner)
print("Planning window (years):", T_planner * p.Delta)


### 3.a) Plots — Optimal Paths


In [ ]:
# Charts rule for this course notebook (matplotlib only; one chart per figure)
import matplotlib.pyplot as plt
import numpy as np

years = sim_opt[:, p.i_time]

# s_t
plt.figure()
plt.plot(years, sim_opt[:, p.i_s], linewidth=2)
plt.title("Optimal saving rate $s_t$")
plt.xlabel("Year"); plt.ylabel("Share"); plt.grid(True)
plt.show()

# K_t
plt.figure()
plt.plot(years, sim_opt[:, p.i_K], linewidth=2)
plt.title("Capital $K_t$")
plt.xlabel("Year"); plt.ylabel("Level"); plt.grid(True)
plt.show()

# Y, C, I (three lines in one figure is acceptable; still one plot figure)
plt.figure()
plt.plot(years, sim_opt[:, p.i_Y], linewidth=2, label="Y")
plt.plot(years, sim_opt[:, p.i_C], linewidth=2, label="C")
plt.plot(years, sim_opt[:, p.i_I], linewidth=2, label="I")
plt.title("Output split: $Y_t = C_t + I_t$")
plt.xlabel("Year"); plt.grid(True); plt.legend()
plt.show()

# c_t per capita
plt.figure()
c = np.full(p.nT, np.nan)
c[1:] = sim_opt[1:, p.i_C] / sim_opt[:-1, p.i_L]
plt.plot(years, c, linewidth=2)
plt.title("Per‑capita consumption $c_t$")
plt.xlabel("Year"); plt.grid(True)
plt.show()

### Capital accumulation and the steady state

The slides and this example use the same baseline: $\alpha=0.30$, $\delta=0.06$, $\rho=0.015$, $\gamma=2$, $A=L=1$ and a one-year step. Initial capital is $K_0=3$.

The steady state satisfies $\alpha (K^*)^{\alpha-1}=\rho+\delta$. The diagram plots the **change in capital**, $K_{t+1}-K_t$, against the capital at the **start** of the same interval, $K_t$. It is a change in levels, not a percentage growth rate.

We use the baseline path below $K^*$ and solve the same model once more with initial capital above $K^*$. All other parameters stay unchanged. Positive changes mean capital accumulates; negative changes mean it decreases. Both paths approach $K^*$.


In [ ]:
# Steady state for the baseline with A = L = 1 and a one-year step.
K_star = (p.alpha / (p.rho + p.delta)) ** (1.0 / (1.0 - p.alpha))
print("Steady-state capital:", K_star)

# Start above the steady state; keep the same default calibration.
p_high = og.Params()
p_high.K0 = 2.0 * K_star
sim_high = og.init_states(p_high)
sim_high = og.run_optimal_policy(
    sim_high, range(1, p_high.nT), p_high, (0.0, 1.0), [p_high.i_s]
)

# Match each change in capital with capital at the start of its interval.
K_low = sim_opt[:-1, p.i_K]
change_low = sim_opt[1:, p.i_K] - sim_opt[:-1, p.i_K]
K_high = sim_high[:-1, p_high.i_K]
change_high = sim_high[1:, p_high.i_K] - sim_high[:-1, p_high.i_K]

plt.figure(figsize=(8, 4.3))
plt.plot(K_low, change_low, linewidth=2, color="tab:blue", label="Initially below the steady state")
plt.plot(K_high, change_high, linewidth=2, color="tab:orange", label="Initially above the steady state")
plt.axhline(0, color="0.35", linewidth=0.8)
plt.axvline(K_star, color="0.45", linestyle="--", linewidth=1)
plt.plot(K_star, 0, "o", color="black", label=r"Steady state $K^*$")
plt.xlabel(r"Capital at the start of the interval, $K_t$")
plt.ylabel(r"Change in capital, $K_{t+1}-K_t$")
plt.title("Capital accumulation and convergence")
plt.legend(fontsize=9)
plt.grid(alpha=0.2)
plt.tight_layout()
# This is the same figure used in the slides.
plt.savefig("images/phase_diagram.pdf", bbox_inches="tight")
plt.show()


## 4) **Exercises** (fill in the code cells)

### Exercise 1 — Reproduce the baseline and check feasibility
- Recreate the baseline path using your own code calls.
- Verify that resource feasibility holds period by period: \(Y_t = C_t + I_t\).


In [ ]:
# TODO: Your code here.
# Hints:
# p1 = og.Params(); ensure p1.lg is defined; sim1 = og.init_states(p1); ...
pass

### Exercise 2 — Compare higher depreciation `delta`
- Increase `delta` by +5 percentage points (e.g., from 0.06 to 0.11).
- Re‑simulate and compare the time paths of \(s_t\), \(K_t\), and \(c_t\).
- Provide a short economic interpretation.


In [ ]:
# TODO: Your code here.
# Example skeleton:
# p_hi = og.Params(); p_hi.delta = 0.11
# sim_hi = og.run_optimal_policy(og.init_states(p_hi), np.arange(1, p_hi.nT, dtype=int), p_hi, (0,1), [p_hi.i_s])
# Plot comparisons vs. baseline.
pass

### Exercise 3 — Compare higher capital share `alpha`
- Increase `alpha` from 0.30 to 0.45.
- Discuss how it changes the optimal savings and capital accumulation.


In [ ]:
# TODO: Your code here.
pass

### Exercise 4 — Add a DICE-style population growth law

So far, we have treated population as an exogenous flat path ($L_t = 1$).  
In integrated assessment models, population follows a **law of motion** that captures the idea of demographic saturation:

$$
L_{t+1} = L_t^{1-\zeta_L} \cdot L_{\infty}^{\zeta_L} ,
$$

- $L_{\infty}$ is the **long-run population level** (carrying capacity),  
- $\zeta_L$ controls the **speed of convergence** toward $L_{\infty}$,  

Questions
- The calibration of $\zeta_L$ is right now 0, calibrate it to 0.02.
- Compare results with stable population.
Use the supplied target Linf=10500 in the same abstract units as L0=1. This is a demographic stress test, not a realistic normalized population forecast.


In [ ]:
# TODO: Your code here.
# You can copy and adapt og.init_states into a new function, then use it in place of the default.
#p.L0   = 1.0           # initial population (scale arbitrary)
#p.Linf = 10500         # Asymptotic population (millions)
#p.lg   = 0             # Population growth rate parameter
pass

#### Interpret your results: does a growing population increase saving? Why?


Write your answer here.


### Exercise 5 — Add DICE-style **TFP** dynamics with declining growth

So far, we kept productivity flat (effectively $A_t \equiv 1$ because `gA = 0`).  
We now introduce **time-varying TFP growth** that declines over time—akin to what DICE uses.

**Law of motion (implemented in `OptimalGrowth.py`):**
$$
A_t \;=\; A_{t-1}\; G_t, 
\qquad\text{with}\qquad 
G_t \;=\; \frac{1}{\,1 - g_A \exp\!\big(-\delta_A\,\Delta\,t\big)\,}\,.
$$

Equivalently, the **per-step proportional growth rate** is approximately
$$
g_{A,t} \;\approx\; g_A \exp\!\big(-\delta_A\,\Delta\,t\big).
$$
Hence, **initial growth** is about $g_A$, then it **decays at rate** $\delta_A$ toward zero.

**Parameters in `Params`:**
- `A0` — initial TFP level (default $A_0 = 1$; normalization).
- `gA` — **initial** TFP growth *per year* (currently set to `0.015*0`, i.e., **zero** in the notebook).
- `deltaA` — **decay rate** of the TFP growth (per year).
- `Delta` — time step (years per period).

---

#### Suggested calibration for this exercise

Use a positive but realistic initial growth, and a slow decay:

- $A_0 = 1$
- $g_A = 0.015$  (≈ **1.5%/year** initial TFP growth)
- $\delta_A = 0.005$  (growth decays at **0.5%/year** toward zero)

**Questions / Tasks**

1. Implement the law of motion above (already in the module) by setting the parameters accordingly.
2. Re-simulate the planner’s problem and **plot $A_t$** and **compare**:
   - saving rate $s_t$,
   - output $Y_t$,
   - consumption per capita $c_t = C_t/L_t$,
   against the baseline with constant TFP.


In [ ]:
# 1) Set parameters for TFP growth that declines over time
p_tfp = og.Params()
p_tfp.A0     = 1.0
p_tfp.gA     = 0.015     # ≈ 1.5%/year initial TFP growth
p_tfp.deltaA = 0.005     # growth decay rate per year (toward 0)

#....
pass

3. Discuss: how does transitional TFP growth alter the optimal saving path and welfare relative to flat TFP?


Write your answer here.
